<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

#第7章练习题答案

## 练习 7.1：更改提示样式

假设我们有以下数据条目：

```json
{
  "instruction": "Identify the correct spelling of the following word.",
  "input": "Ocassion",
  "output": "The correct spelling is 'Occasion.'"
}
```

在主要章节中，我们按照羊驼风格的提示模板进行了格式化：

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Occassion

### Response:
The correct spelling is 'Occasion.'
```

在本练习中，我们现在使用 Phi-3 提示模板，其数据输入格式如下：

```
<user>
Identify the correct spelling of the following word: 'Occasion'

<assistant>
The correct spelling is 'Occasion'.
```

请注意，此提示模板要短得多，这减少了微调 LLM 和生成文本的运行时间和硬件要求，因为输入提示更短。
为了进行此更改，我们更新了 `format_input` 函数，如下所示：

In [1]:
def format_input(entry):
    instruction_text = (
        f"<|user|>\n{entry['instruction']}"
    )

    input_text = f"\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

让我们通过将其应用于两个输入样本（一个在 `'input'` 字段中包含内容，另一个不包含内容）来确保它按预期工作：

In [2]:
sample_data = [
    {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}, 
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}
]

print(format_input(sample_data[0]))
print()
print(format_input(sample_data[1]))

<|user|>
Identify the correct spelling of the following word.
Ocassion

<|user|>
What is an antonym of 'complicated'?


接下来，我们还更新 `InstructionDataset` 类以使用 <|assistant|> 提示模板进行响应：

```python
import tiktoken
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:

            ###################################################################
            # NEW: Use `format_input_phi` and adjust the response text template
            instruction_plus_input = format_input(entry)
            response_text = f"\n<|assistant|>:\n{entry['output']}"
            ###################################################################
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


tokenizer = tiktoken.get_encoding("gpt2")
```

最后，我们还必须更新收集测试集响应时提取生成响应的方式：

```python
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)
    tokenizer=tokenizer

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    # New: Adjust ###Response -> <|assistant|>
    response_text = generated_text[len(input_text):].replace("<|assistant|>:", "").strip()

    test_data[i]["model_response"] = response_text
```

为了您的方便，练习解决方案在 [exercise_experiments.py](exercise_experiments.py) 脚本中实现，您可以按如下方式运行该脚本：

```bash
python exercise_experiments.py --exercise_solution phi3_prompt
```

输出：

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 935
Validation set length: 55
Test set length: 110
--------------------------------------------------
Device: cuda
--------------------------------------------------
...
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Initial losses
   Training loss: 3.71630220413208
   Validation loss: 3.6440994262695314
Ep 1 (Step 000000): Train loss 2.633, Val loss 2.622
...
Ep 2 (Step 000230): Train loss 0.424, Val loss 0.928
<|user|> Convert the active sentence to passive: 'The chef cooks the meal every day.' <|assistant|>: The meal is prepared every day by the chef....
Training completed in 1.50 minutes.
Plot saved as loss-plot-phi3-prompt.pdf
--------------------------------------------------
Generating responses
100% 110/110 [00:11<00:00,  9.27it/s]
Responses saved as instruction-data-with-response-phi3-prompt.json
Model saved as gpt2-medium355M-sft-phi3-prompt.pth
```

为了进行比较，您可以通过 `python exercise_experiments.py --exercise_solution baseline` 运行原始的第 7 章微调代码。 

请注意，在 Nvidia L4 GPU 上，使用 Phi-3 提示模板的上述代码需要 1.5 分钟才能运行。相比之下，Alpaca 样式模板的运行时间为 1.80 分钟。因此，Phi-3 模板的速度大约快 17%，因为它会缩短模型输入。 

让我们看一下一些回复，以确保它们的格式正确：

```json
    {
        "instruction": "Rewrite the sentence using a simile.",
        "input": "The car is very fast.",
        "output": "The car is as fast as lightning.",
        "model_response": "The car is as fast as a cheetah."
    },
    {
        "instruction": "What type of cloud is typically associated with thunderstorms?",
        "input": "",
        "output": "The type of cloud typically associated with thunderstorms is cumulonimbus.",
        "model_response": "The type of cloud associated with thunderstorms is a cumulus cloud."
    },
    {
        "instruction": "Name the author of 'Pride and Prejudice'.",
        "input": "",
        "output": "Jane Austen.",
        "model_response": "The author of 'Pride and Prejudice' is Jane Austen."
    },
```

我们可以使用 Ollama Llama 3 方法来评估性能，为了方便起见，该方法也在 `python exercise_experiments.py` 脚本中实现，我们可以按如下方式运行该方法：

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-phi3-prompt.json
```

输出：

```
Ollama running: True
Scoring entries: 100%|████████████████████████| 110/110 [01:08<00:00,  1.60it/s]
Number of scores: 110 of 110
Average score: 48.87
```

分数接近 50，与我们之前使用羊驼风格提示获得的分数大致相同。

Phi 提示风格没有固有的优势或理由应该更好，但它可以更加简洁和高效，除了下面*提示*部分提到的警告之外。

#### 提示：考虑特殊标记

- 请注意，Phi-3 提示模板包含特殊标记，例如 `<|user|>` 和 `<|assistant|>`，这对于 GPT-2 标记生成器来说可能不是最佳选择
- 虽然 GPT-2 标记生成器将 `<|endoftext|>` 识别为特殊标记（编码为标记 ID 50256），但它在处理其他特殊标记（例如上述标记）时效率较低
- 例如，`<|user|>`被编码成5个单独的令牌ID（27、91、7220、91、29），效率非常低
- 我们可以通过 `allowed_special` 参数将 `<|user|>` 添加为 `tiktoken` 中的新特殊令牌，但请记住，如果不进行额外修改，GPT-2 词汇表将无法处理它
- 如果您对如何扩展分词器和 LLM 来处理特殊代币感到好奇，请参阅 [扩展-tiktoken.ipynb](../../ch05/09_extending-tokenizers/extend-tiktoken.ipynb) 奖励材料（请注意，这不是必需的，但这只是好奇读者的一个有趣/奖励考虑因素）
- 此外，我们可以假设通过词汇支持提示模板的这些特殊标记的模型可能会表现得更高效、整体效果更好

 
## 练习 7.2：指令和输入屏蔽

为了屏蔽掉下图所示的指令，我们需要对`InstructionDataset`类和`custom_collate_fn`进行轻微修改。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/mask-instructions.webp" width=600px>

In [4]:
# This `format_input` function is copied from the original chapter 7 code

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

我们可以修改 `InstructionDataset` 类来收集指令的长度，当我们编写 collat​​e 函数时，我们将在 collat​​e 函数中使用它来定位指令内容在目标中的位置，如下所示：

In [5]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        ##########################################################################################
        # New: Separate list for instruction lengths
        self.instruction_lengths = []
        ##########################################################################################
        
        self.encoded_texts = []
        
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

            ##########################################################################################
            # New: collect instruction lengths
            instruction_length = len(tokenizer.encode(instruction_plus_input))
            self.instruction_lengths.append(instruction_length)
            ##########################################################################################
            
    def __getitem__(self, index):
        # New: return both instruction lengths and texts separately
        return self.instruction_lengths[index], self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [6]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

接下来，我们更新 `custom_collate_fn`，其中由于 `InstructionDataset` 数据集的更改，每个 `batch` 现在是包含 `(instruction_length, item)` 而不是仅 `item` 的元组。此外，我们现在屏蔽目标 ID 列表中相应的指令标记。

In [7]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for instruction_length, item in batch)   # New: batch is now a tuple

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for instruction_length, item in batch:  # New: batch is now a tuple
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        ##########################################################################################
        # New: Mask all input and instruction tokens in the targets
        targets[:instruction_length-1] = -100
        ##########################################################################################
        
        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

让我们在下面的一些示例数据上尝试一下：

In [8]:
sample_data = [
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."},
    {'instruction': 'Sort the following list in alphabetical order.', 'input': 'Zebra, Elephant, Crocodile', 'output': 'Crocodile, Elephant, Zebra'},
    {'instruction': 'Arrange the given numbers in descending order.', 'input': '5, 12, 8, 3, 15', 'output': '15, 12, 8, 5, 3.'}
]

In [9]:
from torch.utils.data import DataLoader

train_dataset = InstructionDataset(sample_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=len(sample_data),
    collate_fn=custom_collate_fn,
    num_workers=0
)

In [10]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([3, 64]) torch.Size([3, 64])


In [11]:
print("Inputs:\n", inputs[1])
print("\n\nTargets:\n", targets[1])

Inputs:
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 42758,   262,  1708,  1351,   287, 24830,
          605,  1502,    13,   198,   198, 21017, 23412,    25,   198,    57,
        37052,    11, 42651,    11,  9325, 19815,   576,   198,   198, 21017,
        18261,    25,   198,    34, 12204,   375,   576,    11, 42651,    11,
         1168, 37052, 50256, 50256])


Targets:
 tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,   198,   198, 21017, 18261,
           25,   198,    34, 12204,   375,   576,    11, 42651,    11,  1168,
      

正如我们基于 `targets` 张量所看到的，指令和填充标记现在都使用 -100 占位符标记进行屏蔽。 
让我们对输入进行解码以确保它们看起来正确：

In [12]:
print(tokenizer.decode(list(inputs[1])))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Sort the following list in alphabetical order.

### Input:
Zebra, Elephant, Crocodile

### Response:
Crocodile, Elephant, Zebra<|endoftext|><|endoftext|>


接下来，让我们解码非屏蔽目标令牌 IDS：

In [13]:
non_masked_targets = targets[1][targets[1] != -100]

print(tokenizer.decode(list(non_masked_targets)))



### Response:
Crocodile, Elephant, Zebra<|endoftext|>


如上所示，未屏蔽的目标令牌按预期排除了 `"Instruction"` 和 `"Input"` 字段。现在，我们可以运行修改后的代码，看看使用此屏蔽策略进行微调时 LLM 的表现如何。

为了方便起见，您可以使用 `exercise_experiments.py` 代码来运行比较，如下所示：

```bash
python exercise_experiments.py --exercise_solution mask_instructions
```

输出：

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 935
Validation set length: 55
Test set length: 110
--------------------------------------------------
Device: cuda
--------------------------------------------------
...
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Initial losses
   Training loss: 2.280539035797119
   Validation loss: 2.262560224533081
Ep 1 (Step 000000): Train loss 1.636, Val loss 1.620
...
Ep 2 (Step 000230): Train loss 0.143, Val loss 0.727
...
Training completed in 1.77 minutes.
Plot saved as loss-plot-mask-instructions.pdf
--------------------------------------------------
Generating responses
100% 110/110 [02:10<00:00,  1.19s/it]
Responses saved as instruction-data-with-response-mask-instructions.json
Model saved as gpt2-medium355M-sft-mask-instructions.pth
```

接下来，让我们评估一下由此产生的LLM的表现：

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-mask-instructions.json
```

```
Ollama running: True
Scoring entries: 100%|██████████████████████████████████████████████████████████████████████████████████████| 110/110 [01:23<00:00,  1.31it/s]
Number of scores: 110 of 110
Average score: 47.73
```

从分数中我们可以看出，指令屏蔽的性能确实稍差一些，这与“Instruction Tuning With Loss Over instructions”论文（https://arxiv.org/abs/2405.14394）中的观察结果一致

 
## 练习 7.3：在原始 Alpaca 数据集上进行微调

要在原始斯坦福羊驼数据集 ([https://github.com/tatsu-lab/stanford_alpaca](https://github.com/tatsu-lab/stanford_alpaca)) 上微调模型，您只需将文件 URL 从

```python
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
```

到

```python
url = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
```

请注意，数据集包含 52k 条目（比第 7 章多 50 倍），并且条目比我们在第 7 章中使用的条目长。
因此，强烈建议在 GPU 上运行训练。

如果遇到内存不足错误，请考虑将批处理大小从 8 减少到 4、2 或 1。除了降低批处理大小之外，您可能还需要考虑将 `allowed_max_length` 从 1024 降低到 512 或 256。

为了方便起见，您可以使用 `exercise_experiments.py` 代码在批量大小为 4、`allowed_max_length` 为 512 的 52k Alpaca 数据集上对模型进行微调，如下所示：

```bash
python exercise_experiments.py --exercise_solution alpaca_52k
```

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 44201
Validation set length: 2601
Test set length: 5200
--------------------------------------------------
Device: cuda
--------------------------------------------------
...
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Initial losses
   Training loss: 3.3681655883789063
   Validation loss: 3.4122894287109373
Ep 1 (Step 000000): Train loss 2.477, Val loss 2.750
...
Ep 2 (Step 022095): Train loss 0.761, Val loss 1.557
...
Training completed in 196.38 minutes.
Plot saved as loss-plot-alpaca52k.pdf
--------------------------------------------------
Generating responses
100% 5200/5200 [2:56:33<00:00,  2.04s/it]
Responses saved as instruction-data-with-response-alpaca52k.json
Model saved as gpt2-medium355M-sft-alpaca52k.pth
```

以下是羊驼数据集的一些示例，包括生成的模型响应：

```json
[
    {
        "instruction": "编辑以下句子以增加可读性：\"他付出了巨大的努力并且非常成功。\"",
        “输入”：“”，
        "output": "他付出了巨大的努力，因此获得了巨大的成功。",
        "model_response": "他付出了巨大的努力并获得了成功。"
    },
    {
        "instruction": "重写以下句子，使其更简洁：\"我对我进行的实验结果感到不满意。\"",
        “输入”：“”，
        "output": "我对实验结果不满意。",
        "model_response": "我对实验结果不满意。"
    },
    {
        "instruction": "我们如何构建更高效的GPT模型？",
        “输入”：“”，
        "output": "我们可以通过优化模型架构、使用更小的模型尺寸和更少的参数进行训练来构建更高效的 GPT 模型。我们还可以利用知识蒸馏、迁移学习、动态稀疏和混合计算等技术来进一步提高模型的效率。",
        "model_response": "构建更高效的 GPT 模型需要仔细规划和优化。首先，确定目标语言和模型使用的上下文很重要。然后，选择适当的模型架构，例如反向传播、超参数和超参数，很重要。最后，选择适当的模型权重和优化器，例如反向传播、超参数和超参数，也很重要。"
    },
]

最后，我们可以使用 [ollama_evaluate.py](ollama_evaluate.py) 实用函数评估微调后的 LLM：

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-alpaca52k.json
```

```
Scoring entries: 100%|████████████████████| 5200/5200 [1:07:52<00:00, 1.28it/s]
Number of scores: 5188 of 5200
Average score: 48.16
```

该分数略低于我们在本章中使用的数据集上获得的分数。然而，请注意，羊驼测试集包含比我们在主要章节中使用的数据集更多样化且部分更具挑战性的指令。

## 练习 7.4：使用 LoRA 进行参数高效的微调

要使用 LoRA 指导微调模型，请使用附录 E 中的相关类和函数：

```python
from appendix_E import LoRALayer, LinearWithLoRA, replace_linear_with_lora
```

接下来，在7.5节的模型加载代码下方添加以下代码行：


```python
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")
replace_linear_with_lora(model, rank=16, alpha=16)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")
model.to(device)
```

为了方便起见，您可以使用 `exercise_experiments.py` 代码来微调模型，使用等级 16 和 alpa 16 的 LoRA，如下所示：

```bash
python exercise_experiments.py --exercise_solution lora
```

输出：

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 935
Validation set length: 55
Test set length: 110
--------------------------------------------------
Device: cuda
--------------------------------------------------
File already exists and is up-to-date: gpt2/355M/checkpoint
File already exists and is up-to-date: gpt2/355M/encoder.json
File already exists and is up-to-date: gpt2/355M/hparams.json
File already exists and is up-to-date: gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/355M/model.ckpt.index
File already exists and is up-to-date: gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: gpt2/355M/vocab.bpe
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Total trainable parameters before: 406,286,336
Total trainable parameters after: 0
Total trainable LoRA parameters: 7,898,384
Initial losses
   Training loss: 3.7684114456176756
   Validation loss: 3.7619335651397705
Ep 1 (Step 000000): Train loss 2.509, Val loss 2.519
...
Ep 2 (Step 000230): Train loss 0.308, Val loss 0.652
...
--------------------------------------------------
Generating responses
100% 110/110 [01:52<00:00,  1.03s/it]
Responses saved as instruction-data-with-response-lora.json
Model saved as gpt2-medium355M-sft-lora.pth
```

为了进行比较，您可以通过 `python exercise_experiments.py --exercise_solution baseline` 运行原始的第 7 章微调代码。 

请注意，在 Nvidia L4 GPU 上，上面使用 LoRA 的代码运行需要 1.30 分钟。相比之下，基线的运行时间为 1.80 分钟。因此，LoRA 的速度大约快 28%。


我们可以使用 Ollama Llama 3 方法来评估性能，为了方便起见，该方法也在 `python exercise_experiments.py` 脚本中实现，我们可以按如下方式运行该脚本：

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-lora.json
```

输出：

```
Ollama running: True
Scoring entries: 100%|████████████████████████| 110/110 [01:13<00:00,  1.50it/s]
Number of scores: 110 of 110
Average score: 50.23
```

分数约为 50 分，与原始模型大致相同。